<img src="../assets/ga-logo.png" style="float: left; margin: 20px; height: 55px">

# Lab: Build a Sentiment Analysis Classifier (Solution)

---

# Objective
In this lab, you will use **CountVectorizer** and **TF-IDF** in a classification model to predict customer star ratings based on their reviews of a product. (This will serve as proxy for sentiment analysis.)
1. Process review text using both CountVectorizer and TF-IDF
2. Build a logistic regression model to predict sentiment
3. Compare the performance of both approaches 

# Scenario 
You are an analyst for a marketing company that has just launched a new product suite of mobile devices. You have data from product reviews of one of these new products, the **TechWave X1**. For this lab you will use the text and star ratings from the reviews to predict customer sentiment. 

# Dataset
- The Reviews data set (`reviews.csv` in `./data/reviews.csv`) is a small sample of reviews created for this Lab


## Step 1: Import Libraries

In [ ]:
%pip install -qqq numpy pandas scikit-learn


In [ ]:
# import numpy, pandas, CountVectorizer, TfidfVectorizer, and the modules from scikit-learn that you need to run a Logistic Regression model.
import numpy as np
import pandas as pd

# NLP imports
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

# model building imports
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix

In [ ]:
# code to avoid truncation of the output below 
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

## Step 2: Load and Preprocess the Data

Use **CountVectorizer** and/or **TF-IDF** to predict the start rating of the reviews for the TechWave X1.

- First import the full data set from `reviews.csv` (`../data/reviews.csv`).
- Then convert the star rating into a binary outcome (0/1) to approximate sentiment analysis.

#### Read in the data and convert the `rating` feature to a numeric

In [ ]:
df_x1 = pd.read_csv('./data/reviews.csv')
df_x1['rating'] = pd.to_numeric(df_x1['rating'])
df_x1.info()

#### Create outcome variable from rating 

You will convert the 5-star rating into a proxy for sentiment where star ratings:  
- 1-3: Negative (0)
- 4-5: Positive (1)

Do this with a lambda function and `.apply()` and create a new sentiment column that will be used as the outcome in our model. 

In [ ]:
# Convert 5-star ratings to binary sentiment
# 1-3: Negative (0), 4-5: Positive (1)

df_x1['sentiment'] = df_x1['rating'].apply(lambda x: 0 if x <= 3 else 1)
pd.crosstab(df_x1['rating'], df_x1['sentiment'])

In [ ]:
df_x1.head()

#### Create a training and test data set 

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    df_x1['review'], df_x1['sentiment'], test_size=0.2, random_state=1212)

#### Process the text with CountVectorizer

In [ ]:
# Initialize a CountVectorizer, fit and transform the training data, and transform the testing data
vectorizer = CountVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the training data
X_train_transformed = vectorizer.fit_transform(X_train)

# Transform the testing data
X_test_transformed = vectorizer.transform(X_test)

## Step 3: Build a logistic model to predict sentiment using the processed review text

Build a logistic regression model using the words as features to predict sentiment/ratings.

Checkout the [scikit-learn logistic regression docs](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html) if you need a reminder of how to create a **Logistic Regression** model.


##### Hint!
```python
# Initialize a Logistic Regression model
model = LogisticRegression()
```

In [ ]:
# Initialize a Logistic Regression model, train the model, predict the ratings, evaluate the model using accuracy, and show the confusion matrix.
model = LogisticRegression()

# Train the model on the transformed training data
model.fit(X_train_transformed, y_train)

# Predict the ratings for the testing data
y_pred = model.predict(X_test_transformed)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Display the confusion matrix
print(confusion_matrix(y_test, y_pred))

#### Repeat the process with TF-IDF.  Process the text with TF-IDF

In [ ]:
# Initialize TfidfVectorizer
vectorizer = TfidfVectorizer(ngram_range=(1, 2), stop_words='english')

# Fit and transform the training data
X_train_transformed = vectorizer.fit_transform(X_train)

# Transform the testing data
X_test_transformed = vectorizer.transform(X_test)

#### Build a Logistic Regression model with TF-IDF processed text

In [ ]:
# Initialize a Logistic Regression model
model = LogisticRegression()

# Train the model on the transformed training data
model.fit(X_train_transformed, y_train)

# Predict the ratings for the testing data
y_pred = model.predict(X_test_transformed)

# Evaluate the model's performance
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

# Display the confusion matrix
print(confusion_matrix(y_test, y_pred))

## Stretch
Here we have used Logistic Regression, but other classification models such as Naive Bayes, Random Forest, or SVM, may perform better. Feel free to explore if you would like extra practice.

# Step 4: What did you observe? 

CountVectorizer achieved slightly higher accuracy than TF-IDF on our test data.

Looking at the confusion matrices:
- CountVectorizer correctly identified 2 negative reviews and all positive reviews
- TF-IDF classified all reviews as positive

This suggests that the raw word frequencies captured by CountVectorizer were more informative for our task than the weighted frequencies from TF-IDF. This makes sense because if someone uses negative words multiple times in a review, that repetition itself may be meaningful for sentiment, rather than something that should be downweighted as is the case with TF-IDF.